# SQL数据交换

学习目标：在 DataFrame 与 SQLite 之间读写数据，明确查询参数、类型、日期、写入方式和事务边界，并正确释放连接。

前置知识：SQL SELECT、WHERE、表与主键，连接、文件操作和资源关闭。

运行环境：Python 3.12、pandas 3；SQLite 使用 Python 标准库 sqlite3，事务示例显式设置 Python 3.12 新增的 autocommit 参数。

环境准备：[环境配置与运行](README.md)

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

本章使用自制小表和临时数据库，不连接外部服务。临时文件在示例结束时清理。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 写入订单并查询

to_sql 将 DataFrame 写入数据库表，read_sql_query 将查询结果读为 DataFrame。下面先保存三条订单，再查出数量至少为 2 的记录；order_id 是保留前导零的文本编号，qty 的单位是件。

TemporaryDirectory 创建临时目录；内层 closing 在退出时关闭连接，外层随后删除目录。index=False 表示不把 DataFrame 的行索引写成数据库列。

In [1]:
import sqlite3
from contextlib import closing
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

orders = pd.DataFrame({"order_id": ["001", "002", "003"], "qty": [2, 1, 5]})
with TemporaryDirectory() as temp_dir:
    database_path = Path(temp_dir) / "orders.sqlite"
    with closing(sqlite3.connect(database_path, autocommit=False)) as con:
        orders.to_sql("orders", con, if_exists="fail", index=False)
        selected = pd.read_sql_query(
            "SELECT order_id, qty FROM orders WHERE qty >= ? ORDER BY order_id",
            con, params=(2,),
        )
        print(selected)  # 预期：两行 order_id 为 001、003，qty 分别为 2、5。
        print(selected.dtypes)
print("临时数据库已删除：", not database_path.exists())
# 返回 001、003，数量为 2、5；编号是 str，数量是 int64；最后输出 True。
assert not Path(temp_dir).exists()

  order_id  qty
0      001    2
1      003    5
order_id      str
qty         int64
dtype: object
临时数据库已删除： True


## 2 参数绑定与结果顺序

上例的 ? 是值的占位符，params=(2,) 把数值交给 SQLite 驱动绑定。不要用字符串拼接或格式化把输入值写进 SQL；即使名字带单引号，参数绑定也能把它当作普通文本处理。

SQLite 也支持 :name 形式的命名占位符，此时 params 使用字典。占位符只绑定值，不能替代表名、列名或 SQL 关键字；本章把这些结构固定写在代码中。

SQL 结果没有默认的行顺序。需要稳定结果时显式写 ORDER BY，并使用能够区分记录的排序键。以下各独立示例使用 :memory: 内存数据库，连接关闭后不保留数据。

In [2]:
people = pd.DataFrame({"id": [2, 1], "name": ["Lin", "O'Neil"]})
with closing(sqlite3.connect(":memory:", autocommit=False)) as con:
    people.to_sql("people", con, index=False)
    found = pd.read_sql_query(
        "SELECT id, name FROM people WHERE name = :name ORDER BY id",
        con, params={"name": "O'Neil"},
    )
    print(found)  # 预期：一行 id=1、name=O'Neil，单引号作为参数内容保留。
    print(pd.read_sql_query(
        "SELECT id FROM people WHERE name = ? ORDER BY id",
        con, params=("' OR 1=1 --",),
    ).shape)
# O'Neil 正确匹配 id=1；看似 SQL 的第二个输入仍是值，匹配结果为 (0, 1)。

   id    name
0   1  O'Neil
(0, 1)


## 3 SQL 类型与缺失值

SQL 列类型与 pandas dtype 是两层约定。SQLite 常见存储类型如下；数据库不会保存 pandas 的扩展 dtype 元信息。

| SQLite 类型 | 中文名称／含义 | sqlite3 默认读取结果 |
| --- | --- | --- |
| INTEGER | 整数 | Python int |
| REAL | 浮点数 | Python float |
| TEXT | 文本 | Python str |
| BLOB | 二进制数据 | Python bytes |
| NULL | 缺失值 | Python None |

to_sql 的 dtype 指定建表时的 SQL 列类型，在 sqlite3 连接下使用 "TEXT"、"INTEGER" 等字符串。read_sql_query 的 dtype 则指定读回的 pandas 类型。两者不能混用。

In [3]:
stock = pd.DataFrame({
    "sku": ["001", "002", "003"],
    "qty": pd.Series([2, pd.NA, 5], dtype="Int64"),
})
with closing(sqlite3.connect(":memory:", autocommit=False)) as con:
    stock.to_sql(
        "stock", con, index=False, dtype={"sku": "TEXT", "qty": "INTEGER"},
    )
    inferred = pd.read_sql_query("SELECT * FROM stock ORDER BY sku", con)
    restored = pd.read_sql_query(
        "SELECT * FROM stock ORDER BY sku", con, dtype={"qty": "Int64"},
    )
    print(inferred)  # 预期：sku 为 001/002/003，qty 为 2.0/NaN/5.0。
    print(inferred.dtypes)  # 预期：sku 为 str，qty 为 float64。
    print(restored)  # 预期：sku 不变，qty 为 2/<NA>/5，缺失整数用 Int64 表示。
    print(restored.dtypes)
    pd.testing.assert_frame_equal(stock, restored)
# 默认读取的 qty 为 float64，缺失为 NaN；指定 Int64 后恢复为整数与 <NA>。
# 前导零仍在，行列顺序、值、dtype 均与 stock 一致。

   sku  qty
0  001  2.0
1  002  NaN
2  003  5.0
sku        str
qty    float64
dtype: object
   sku   qty
0  001     2
1  002  <NA>
2  003     5
sku      str
qty    Int64
dtype: object


在 SQL 中查找缺失使用 IS NULL，而不是 = NULL。下面继续使用 stock 作为写入输入，缺失数量读到 Python 时对应 None，读到 pandas 后由目标 dtype 决定缺失标记。

In [4]:
with closing(sqlite3.connect(":memory:", autocommit=False)) as con:
    stock.to_sql("stock", con, index=False)
    # 预期：(None,)，SQL NULL 由 sqlite3 读成 None。
    print(con.execute("SELECT qty FROM stock WHERE sku = ?", ("002",)).fetchone())
    print(pd.read_sql_query("SELECT sku FROM stock WHERE qty IS NULL", con))
    print(pd.read_sql_query("SELECT sku FROM stock WHERE qty = NULL", con).shape)
# 原始读取为 (None,)；IS NULL 找到 002；= NULL 不能匹配该行。

(None,)
   sku
0  002
(0, 1)


SQLite 普通表采用类型亲和性（type affinity），列声明通常表达转换偏好，不等于严格的类型校验。例如 INTEGER 列仍能保存无法转成数字的文本；需要额外约束或严格表时，应另行设计数据库结构。

to_sql 的 dtype 也不会修改已存在表的结构。向已有表追加数据时，真正的限制来自数据库中已有的列和约束。

In [5]:
with closing(sqlite3.connect(":memory:", autocommit=False)) as con:
    con.execute("CREATE TABLE flexible (qty INTEGER)")
    con.execute("INSERT INTO flexible (qty) VALUES (?)", ("unknown",))
    con.commit()
    print(pd.read_sql_query("SELECT qty, typeof(qty) AS storage FROM flexible", con))
# 列声明为 INTEGER，但此值的实际存储类型为 text；声明本身没有拒绝它。

       qty storage
0  unknown    text


## 4 读取日期列

SQLite 没有单独的日期时间存储类型。本章把日期保存为约定格式的文本，读取时通过 parse_dates 显式解析。这样不依赖 Python 3.12 已弃用的默认日期适配器和转换器。

parse_dates 可以把列名映射到格式，也可以映射到 to_datetime 的参数字典。下面将原文本保留为 created_text，同时生成 created 日期列；errors="coerce" 把非法日期转为 NaT。

In [6]:
events = pd.DataFrame({
    "id": [1, 2, 3], "created_text": ["2026-09-20", "bad", None],
})
with closing(sqlite3.connect(":memory:", autocommit=False)) as con:
    events.to_sql("events", con, index=False)
    dated = pd.read_sql_query(
        "SELECT id, created_text, created_text AS created FROM events ORDER BY id",
        con, parse_dates={"created": {"format": "%Y-%m-%d", "errors": "coerce"}},
    )
dated["failed"] = dated["created_text"].notna() & dated["created"].isna()
print(dated)
print(dated["created"].dtype)
# id=2 解析失败，id=3 原本缺失；created 为 datetime64[us]，不自动恢复原意。

   id created_text    created  failed
0   1   2026-09-20 2026-09-20   False
1   2          bad        NaT    True
2   3          NaN        NaT   False
datetime64[us]


数字时间戳还需要单位和起点。下面约定 event_second 是从 Unix 起点计算的秒数，parse_dates 通过 unit="s" 和 utc=True 将它读为 UTC 时间。SQLite 中的整数并没有自带这些含义，必须由数据约定提供。

In [7]:
ticks = pd.DataFrame({"id": [1, 2], "event_second": [0, 60]})
with closing(sqlite3.connect(":memory:", autocommit=False)) as con:
    ticks.to_sql("ticks", con, index=False)
    timed = pd.read_sql_query(
        "SELECT * FROM ticks ORDER BY id", con,
        parse_dates={"event_second": {"unit": "s", "origin": "unix", "utc": True}},
    )
print(timed)
print(timed["event_second"].dtype)
# 1970-01-01 00:00、00:01 UTC；结果单位为 s，并带 UTC 时区。

   id              event_second
0   1 1970-01-01 00:00:00+00:00
1   2 1970-01-01 00:01:00+00:00
datetime64[s, UTC]


## 5 行索引如何保存

to_sql 默认 index=True，会把 DataFrame 索引写成一列，并为该列创建数据库索引。数据库索引用于查询组织，不等于自动声明主键或唯一性。

如果标签确实需要保存，用 index_label 明确列名；读取时用 index_col 将它还原为行索引。仅为临时行号的索引通常不必写入。

In [8]:
labeled = pd.DataFrame({"qty": [2, 5]}, index=pd.Index(["A", "B"], name="batch"))
with closing(sqlite3.connect(":memory:", autocommit=False)) as con:
    labeled.to_sql("batches", con, index=True, index_label="batch")
    print(pd.read_sql_query("SELECT * FROM batches ORDER BY batch", con))  # 预期：两行 batch=A/B、qty=2/5，batch 此时是普通列。
    round_trip = pd.read_sql_query(
        "SELECT * FROM batches ORDER BY batch", con, index_col="batch",
    )
    print(round_trip)  # 预期：qty 为 2/5，行索引恢复为命名的 batch，标签为 A/B。
    print(round_trip.index.name)
    pd.testing.assert_frame_equal(labeled, round_trip)
# 普通读取时 batch 是列；index_col 指定后成为名为 batch 的索引。

  batch  qty
0     A    2
1     B    5
       qty
batch     
A        2
B        5
batch


## 6 已有表的写入方式

if_exists 决定同名表存在时怎么处理。以下都是表级动作，append 不会按编号自动更新或去重。

| 参数值 | 中文名称／含义 |
| --- | --- |
| fail | 拒绝覆盖，抛出 ValueError；默认值 |
| append | 向现有表插入新行 |
| replace | 删除整张表，再根据输入新建并写入 |
| delete_rows | pandas 3 支持的清空行后写入，保留已有表结构 |

In [9]:
one = pd.DataFrame({"id": [1], "qty": [2]})
with closing(sqlite3.connect(":memory:", autocommit=False)) as con:
    one.to_sql("items", con, index=False)
    try:
        one.to_sql("items", con, if_exists="fail", index=False)
    except ValueError:
        print("ValueError：表已存在")  # 预期：if_exists="fail" 进入此分支，显示表已存在的提示。
    else:
        raise AssertionError("fail 应拒绝同名表")
    one.to_sql("items", con, if_exists="append", index=False)
    print(pd.read_sql_query("SELECT * FROM items ORDER BY id", con))
# append 后出现两条 id=1；to_sql 创建的这张表没有主键约束。

ValueError：表已存在
   id  qty
0   1    2
1   1    2


如果需要约束，先用 SQL 建表，再追加符合规则的数据。这里 id 明确声明为主键。pandas 3.0.6 的 SQLite 写入会把底层插入错误包装为 pandas.errors.DatabaseError，可从异常原因检查具体失败类型。

In [10]:
with closing(sqlite3.connect(":memory:", autocommit=False)) as con:
    con.execute("CREATE TABLE items (id INTEGER PRIMARY KEY, qty INTEGER NOT NULL)")
    con.commit()
    one.to_sql("items", con, if_exists="append", index=False)
    try:
        one.to_sql("items", con, if_exists="append", index=False)
    except pd.errors.DatabaseError as exc:
        print(type(exc).__name__, type(exc.__cause__).__name__)  # 预期：DatabaseError IntegrityError，分别是外层异常与直接原因。
        assert isinstance(exc.__cause__, sqlite3.IntegrityError)
    else:
        raise AssertionError("主键应拒绝重复编号")
    print(pd.read_sql_query("SELECT * FROM items ORDER BY id", con))
# 重复编号触发 IntegrityError，外层为 DatabaseError；原来的 1 行仍在。

DatabaseError IntegrityError
   id  qty
0   1    2


replace 的删除重建会丢失原表的主键等约束及其索引、触发器；传入的数据列不能完整描述原数据库结构。delete_rows 保留表结构，但会删除全部旧行，新输入仍须满足原约束。

PRAGMA table_info 可查看普通列的定义：pk 为 0 表示不在主键中，大于 0 表示在主键中的位置。下面只比较主键是否保留。

In [11]:
replacement = pd.DataFrame({"id": [2], "qty": [9]})
with closing(sqlite3.connect(":memory:", autocommit=False)) as con:
    con.execute("CREATE TABLE items (id INTEGER PRIMARY KEY, qty INTEGER NOT NULL)")
    con.commit()
    one.to_sql("items", con, if_exists="append", index=False)
    replacement.to_sql("items", con, if_exists="delete_rows", index=False)
    print(pd.read_sql_query("SELECT * FROM items", con))  # 预期：仅一行 id=2、qty=9。
    # 预期：首次 id 的 pk=1；replace 重建表后 id、qty 的 pk 均为 0。
    print(pd.read_sql_query("PRAGMA table_info(items)", con)[["name", "pk"]])
    replacement.to_sql("items", con, if_exists="replace", index=False)
    # 预期：首次 id 的 pk=1；replace 重建表后 id、qty 的 pk 均为 0。
    print(pd.read_sql_query("PRAGMA table_info(items)", con)[["name", "pk"]])
# delete_rows 后只剩 id=2，id 的 pk=1；replace 重建后 id 的 pk=0。

   id  qty
0   2    9
  name  pk
0   id   1
1  qty   0
  name  pk
0   id   0
1  qty   0


## 7 事务与连接关闭

Python 3.12 的 sqlite3 支持 autocommit 参数。本章明确设为 False：用 commit 提交，用 rollback 撤销尚未提交的事务；不依赖当前默认的旧版事务模式。

对普通 sqlite3 操作，with con 正常退出时提交，异常离开时回滚。它不会关闭连接。外层 closing 负责关闭，两者职责不同。

下面先提交基准数据，再在同一事务中更改数量并故意插入重复主键。异常必须离开 with con，才能触发该事务的回滚。

In [12]:
with closing(sqlite3.connect(":memory:", autocommit=False)) as con:
    con.execute("CREATE TABLE inventory (id INTEGER PRIMARY KEY, qty INTEGER)")
    with con:
        con.execute("INSERT INTO inventory VALUES (?, ?)", (1, 5))
    # 异常先离开事务块以触发回滚，再捕获它以便继续查询并观察回滚后的数量。
    try:
        with con:
            con.execute("UPDATE inventory SET qty = ? WHERE id = ?", (4, 1))
            con.execute("INSERT INTO inventory VALUES (?, ?)", (1, 9))
    except sqlite3.IntegrityError:
        print("IntegrityError：本事务已回滚")  # 预期：主键冲突进入此分支，显示事务回滚提示。
    else:
        raise AssertionError("应触发主键冲突")
    print(con.execute("SELECT qty FROM inventory WHERE id = ?", (1,)).fetchone())
    # 离开 with con 后仍能查询；数量恢复为基准值 5。

# 预期 ProgrammingError：离开外层 closing 后连接已关闭，不能继续执行查询。
con.execute("SELECT 1")

IntegrityError：本事务已回滚
(5,)


ProgrammingError: Cannot operate on a closed database.

## 8 to_sql 的回滚边界

pandas 官方说明：向 sqlite3.Connection 使用 to_sql 写入，不能由调用方回滚已完成的记录插入。不能把上面的纯 sqlite3 事务保证直接套到多个 to_sql 调用上。

以下写入成功后再调用 rollback，数据仍在。需要多个写入步骤共同提交或共同撤销时，应明确选择支持该事务安排的连接方式；只在外面套 with con 不足以改变 to_sql 的内部提交行为。

In [13]:
with closing(sqlite3.connect(":memory:", autocommit=False)) as con:
    one.to_sql("saved", con, index=False)
    con.rollback()
    still_saved = pd.read_sql_query("SELECT * FROM saved ORDER BY id", con)
    print(still_saved)
    pd.testing.assert_frame_equal(one, still_saved)
# rollback 后仍有 id=1、qty=2；不能把它当作撤销本次 to_sql 的办法。

   id  qty
0   1    2


## 9 选学：分块读取与批次写入
read_sql_query 的 chunksize 指定每块最多多少行，返回可逐块读取的迭代器。迭代应在连接仍打开时完成。按块处理后只保留汇总值，才能避免重新积累全部数据；这里不把各块收集成列表。

下面继续使用 orders。SQL 显式排序，保证检查每块成员时有确定的顺序。

In [14]:
with closing(sqlite3.connect(":memory:", autocommit=False)) as con:
    orders.to_sql("orders", con, index=False)
    row_count = 0
    total_qty = 0
    for chunk in pd.read_sql_query(
        "SELECT * FROM orders ORDER BY order_id", con, chunksize=2,
    ):
        print(chunk["order_id"].tolist())  # 预期：依次为 ['001', '002']、['003']。
        row_count += len(chunk)
        total_qty += int(chunk["qty"].sum())
    print(row_count, total_qty)
# 两块成员为 001、002 和 003；总计 3 行、8 件。

['001', '002']
['003']
3 8


to_sql 的 chunksize 控制每批写入的行数；method="multi" 在一条 INSERT 中放多组值。它们改变写入批次和语句组织，不保证更快，也不表示每批都是调用方可独立控制的事务。

不同数据库对 multi 的支持和语句参数数量限制不同。下面只验证 SQLite 的三行小例，并通过读回核对结果，不把返回的影响行数作为唯一成功依据。

In [15]:
with closing(sqlite3.connect(":memory:", autocommit=False)) as con:
    written = orders.to_sql("orders", con, index=False, chunksize=2, method="multi")
    checked = pd.read_sql_query("SELECT * FROM orders ORDER BY order_id", con)
    print(written, checked.shape)  # 预期：3 (3, 2)，写入三行并读回两列。
    print(checked)
    pd.testing.assert_frame_equal(orders, checked)
# 此例报告写入 3 行，读回形状为 (3, 2)，且值、标签顺序与 dtype 均通过比较。

3 (3, 2)
  order_id  qty
0      001    2
1      002    1
2      003    5


## 10 选学：外部数据库入口
read_sql_query 可以通过 SQLAlchemy 连接其支持的数据库；通常还需要该数据库的驱动。占位符样式、权限、类型和事务规则要按实际驱动核查，不能把 SQLite 示例的 ? 原样套到所有数据库。

read_sql_table 按表名读取，要求 SQLAlchemy 连接或连接字符串，不支持直接传入 sqlite3 连接。只用标准库时，继续使用 read_sql_query 和明确的 SELECT。

to_sql 接收一个已处于事务中的 SQLAlchemy Connection 时，不会替调用方提交该事务；这与本章 sqlite3 连接的行为不同。连接关闭和 Engine 资源释放仍须由使用者负责。本章不安装 SQLAlchemy、不连接外部数据库，也不把这些入口说明视为已完成外部数据库实验。

## 本章小结

（1）to_sql 写表，read_sql_query 读查询结果；查询值通过 params 绑定，结果顺序通过 ORDER BY 约定。

（2）SQL 类型与 pandas dtype 分别控制；缺失和日期应按约定读回，并检查原值、键、行数与类型。

（3）append 是插入，replace 是删表重建，delete_rows 是清空记录。需要保留的主键和约束不能只靠 DataFrame 描述。

（4）事务结束与连接关闭是两件事；纯 sqlite3 的回滚示例不能证明多个 to_sql 调用具有整体回滚保证。

## 练习

（1）把下面的库存表写入临时 SQLite 数据库，使用参数绑定查询数量至少为 3 的记录，并按 sku 排序。结束后确认连接关闭、临时文件已删除。

In [16]:
practice_stock = pd.DataFrame({"sku": ["010", "002", "030"], "qty": [4, 1, 3]})
minimum = 3
# 补充：用 TemporaryDirectory 和 closing 管理资源，index=False 写入。
# 检查：只返回 010、030，前导零保留；查询使用 params，数据库文件已清理。

（2）先预测两次读取的 qty dtype 和缺失标记，再运行代码。解释为何 SQLite INTEGER 不保证默认读回仍是 pandas 整数列。

In [17]:
prediction = pd.DataFrame({"qty": pd.Series([1, pd.NA], dtype="Int64")})
with closing(sqlite3.connect(":memory:", autocommit=False)) as con:
    prediction.to_sql("counts", con, index=False)
    first = pd.read_sql_query("SELECT qty FROM counts ORDER BY qty", con)
    second = pd.read_sql_query(
        "SELECT qty FROM counts ORDER BY qty", con, dtype={"qty": "Int64"},
    )
    print(first)
    print(first.dtypes)
    print(second)
    print(second.dtypes)
# 补充：在代码注释中记录运行前预测和运行后的解释。

   qty
0  NaN
1  1.0
qty    float64
dtype: object
    qty
0  <NA>
1     1
qty    Int64
dtype: object


（3）原任务只要求覆盖临时表，现在改为“替换全部记录，但必须保留已建立的主键约束”。你会选择 replace 还是 delete_rows？说明理由，并写一个最小例子检查 pk 标记。若要求只追加新记录，方法又应怎样改变？

In [18]:
new_inventory = pd.DataFrame({"id": [2, 3], "qty": [7, 8]})
# 补充：在 closing 内用 CREATE TABLE 建立 id 主键，写入并选择合适的方式。
# 检查：最终只有 2、3 两个编号，PRAGMA table_info 中 id 的 pk 仍为 1。
# 说明：追加不等于更新或去重，重复编号应如何检查？

（4）有人计划用 with con 包住两个 to_sql 调用，认为第二次失败时第一次一定会撤销。请结合本章实际连接类型判断这句话，并设计一个只用普通 sqlite3 SQL 操作的两步事务：第二步主键冲突后，第一步的修改也应撤销。

In [19]:
baseline_qty = 5
changed_qty = 4
# 补充：使用 autocommit=False，先提交基准行，再在 with con 内执行两步。
# 检查：捕获离开事务块后的 IntegrityError，数量仍为 5；关闭连接。
# 说明：上述成功回滚为何不能证明 sqlite3 连接的 to_sql 也有同样保证？

### 重点练习提示（第 3 题）

提示一：先确认需要保留的是表结构与约束，还是只保留表名。

提示二：先用 CREATE TABLE 建立 id INTEGER PRIMARY KEY，再选择只清空数据的写入方式。

### 参考解析（第 3 题）

本题选择 to_sql 的 if_exists="delete_rows"、index=False：删除现有行后写入 id 为 2、3 的数据，保留已建表结构；PRAGMA table_info 的 id 行 pk 应为 1。replace 会先删除表再重建，不能据此承诺保留原主键。只追加时选择 append，但它不负责更新或去重；已有主键与新增数据重复时应报完整性错误。测试在临时 SQLite 中、closing 管理的连接内完成，并在退出后检查清理；这个结构保留结论不等于承诺 sqlite3 连接的 to_sql 可被外层事务完整回滚。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方文档 | 核查版本 pandas 3.0.6。[read_sql_query](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html) 的 con、params、index_col、dtype、parse_dates、chunksize；[DataFrame.to_sql](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html) 的 con 事务条件、if_exists、index、index_label、dtype、chunksize、method、返回行数与 Notes；[read_sql_table](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_table.html) 的连接要求；[IO tools：SQL queries](https://pandas.pydata.org/docs/user_guide/io.html#sql-queries) 的连接与驱动入口。异常包装及提交另核查本环境官方发行包 pandas 3.0.6 的 pandas/io/sql.py：SQLiteTable._execute_insert、SQLiteTable._create_table_setup、SQLTable.create、SQLiteDatabase.run_transaction。 |
| Python 官方文档 | Python 3.12.14 [sqlite3](https://docs.python.org/3.12/library/sqlite3.html)：How to use placeholders to bind values in SQL queries、SQLite and Python types、Default adapters and converters (deprecated)、How to use the connection context manager、Transaction control via the autocommit attribute，以及 IntegrityError、ProgrammingError；[contextlib.closing](https://docs.python.org/3.12/library/contextlib.html#contextlib.closing) 与 [TemporaryDirectory](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory)：异常路径下的关闭与清理。 |
| SQLite 官方文档 | [Datatypes](https://www.sqlite.org/datatype3.html) 的 Storage Classes、Date and Time Datatype、Type Affinity；[SELECT §4 ORDER BY](https://www.sqlite.org/lang_select.html#orderby) 的结果顺序；[Expressions](https://www.sqlite.org/lang_expr.html) 的 Parameters、NULL 与 IS、IS NOT；[CREATE TABLE](https://www.sqlite.org/lang_createtable.html) 的 PRIMARY KEY、NOT NULL；[DROP TABLE](https://www.sqlite.org/lang_droptable.html) 的表及关联对象删除；[CREATE INDEX](https://www.sqlite.org/lang_createindex.html#uniqueidx) 的普通索引与唯一索引；[PRAGMA table_info](https://www.sqlite.org/pragma.html#pragma_table_info) 的列定义与 pk；[Core functions](https://www.sqlite.org/lang_corefunc.html#typeof) 的 typeof；本章 SQLite 行为以 Python 所带的运行库实际输出核对。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[io](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/io.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |